In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import mean_squared_error as mse, mean_absolute_error as mae, r2_score

In [2]:
df = pd.read_csv('Data.csv')
df.head()

,Input 1,Input 2,Input 3,Output
0,1.0,30,30,4954.505064
1,1.0,30,35,4218.041602
2,1.0,30,40,3563.425156
3,1.0,30,45,3010.799084
4,1.0,30,50,2553.220128


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1065 entries, 0 to 1064
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Input 1  1065 non-null   float64
 1   Input 2  1065 non-null   int64  
 2   Input 3  1065 non-null   int64  
 3   Output   1064 non-null   float64
dtypes: float64(2), int64(2)
memory usage: 33.4 KB


In [4]:
df.dropna(inplace = True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1064 entries, 0 to 1064
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Input 1  1064 non-null   float64
 1   Input 2  1064 non-null   int64  
 2   Input 3  1064 non-null   int64  
 3   Output   1064 non-null   float64
dtypes: float64(2), int64(2)
memory usage: 41.6 KB


In [5]:
X = df.drop(columns = 'Output')
y = df['Output']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)

In [7]:
params_grid = {
    'n_estimators': [50, 75, 100, 150, 200, 250],
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'max_depth': [4, 6, 8, 12],
    'reg_alpha': [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
    'reg_lambda': [0.1, 0.5, 1.0, 2.0]
}

In [8]:
model = XGBRegressor(objective = 'reg:squarederror', device = 'cuda')

In [12]:
gridsearch = GridSearchCV(model, params_grid, cv = 3, scoring = 'neg_mean_squared_error')

In [13]:
gridsearch.fit(X_train, y_train)

c:\Users\rajee\miniconda3\envs\tfenv\lib\site-packages\xgboost\core.py:774: UserWarning: [23:05:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


KeyboardInterrupt: 

In [11]:
bestmodel = gridsearch.best_estimator_
bestparams = gridsearch.best_params_

In [12]:
bestmodel, bestparams

(XGBRegressor(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.3, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=250,
              n_jobs=None, num_parallel_tree=None, ...),
 {'learning_rate': 0.3,
  'max_depth': 4,
  'n_estimators': 250,
  'reg_alpha': 0.2,
  'reg_lambda': 0.5})

In [13]:
print(mse(y_test, bestmodel.predict(X_test)))
print(r2_score(y_test, bestmodel.predict(X_test)))

82.10055883434103
0.9998555327686066


In [14]:
gscv_results = pd.DataFrame(gridsearch.cv_results_)

In [15]:
gscv_results.to_csv('GSCV_Results-multithreaded.csv')

In [ ]:
# import pickle

# with open('Models/XGBR-GSCV.pkl', 'wb') as f:
#     pickle.dump(bestmodel, f)

In [16]:
res_df = df.copy()

In [17]:
res_df['XGBR-GSCV'] = bestmodel.predict(X)

res_df.to_csv('XGBR-GSCV.csv')